# Baselines

### Setup and Imports


In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import json
import ast
import time
import traceback
from tqdm import tqdm
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Add paths for imports
sys.path.append('../rq3_1_Embedding_based_space')  # for RAGSourceFinder
sys.path.append('../../utils')  # for easy_llm_importer

from rag_source_finder import RAGSourceFinder
from easy_llm_importer import create_client, LLMClient, DSPyAdapter

import dspy

# =============================================================================
# CONFIGURATION
# =============================================================================
DEBUG_VERBOSE = True
ERROR_MARKER = "ERROR"
DATA_PATH = "../../dataset/SCAR_cleaned_manually.csv"

# Initialize DSPy with LLM
client = create_client()  # Create the client first
MODEL_NAME = "llama-3.1-405b-instruct"
dspy_adapter = DSPyAdapter(client, model_name=MODEL_NAME)  # Pass both client and model_name
dspy_lm = dspy_adapter.get_dspy_lm()
dspy.configure(lm=dspy_lm)

print("Setup complete!")


Setup complete!


### Baseline 1: Pure Embedding Retrieval (Top 3)


In [3]:
# Initialize RAG source finder
finder = RAGSourceFinder(embedding_mode="name_properties")
finder.load_corpus_from_csv(DATA_PATH)
finder.embed_corpus()


Loaded 333 unique sources from corpus (mode: name_properties)
Generating embeddings for corpus...
Generated 333 embeddings


In [ ]:
# Run Baseline 1: Pure embedding retrieval with top 3
baseline1_results = finder.evaluate_on_dataset(DATA_PATH, top_k=3)
baseline1_results['baseline'] = 'embedding_top3'
baseline1_results.to_csv('results/pure_embedding_top3.csv', index=False)
print(f"Baseline 1 saved: {len(baseline1_results)} rows")
baseline1_results.head()


Evaluating on 400 examples...


Processed 10/400 examples
Processed 20/400 examples
Processed 30/400 examples
Processed 40/400 examples
Processed 50/400 examples
Processed 60/400 examples
Processed 70/400 examples
Processed 80/400 examples
Processed 90/400 examples
Processed 100/400 examples
Processed 110/400 examples
Processed 120/400 examples
Processed 130/400 examples
Processed 140/400 examples
Processed 150/400 examples
Processed 160/400 examples
Processed 170/400 examples
Processed 180/400 examples
Processed 190/400 examples
Processed 200/400 examples
Processed 210/400 examples
Processed 220/400 examples
Processed 230/400 examples
Processed 240/400 examples
Processed 250/400 examples
Processed 260/400 examples
Processed 270/400 examples
Processed 280/400 examples
Processed 290/400 examples
Processed 300/400 examples
Processed 310/400 examples
Processed 320/400 examples
Processed 330/400 examples
Processed 340/400 examples
Processed 350/400 examples
Processed 360/400 examples
Processed 370/400 examples
Processed 

,id,target,gold_source,all_golden_sources,predicted_rank_1,gold_rank,num_golden_found,found_golden_sources,golden_ranks,top_k_sources,top_k_scores,embedding_mode,query_embedding_text,target_background,target_properties,baseline
0,1,biological clock,clock,[clock],clock,1,1,[clock],[1],"[clock, Biological Evolution, Time Machine]","[0.48744014145121656, 0.4727238510932971, 0.43...",name_properties,"biological clock. changes, state, adjust",The biological clock is a fundamental aspect o...,"changes, state, adjust",embedding_top3
1,2,Biosphere,Library,[Library],Ecosystem,-1,0,[],[],"[Ecosystem, ecosystem, Biological Evolution]","[0.7418378212352338, 0.6007781199547637, 0.463...",name_properties,"Biosphere. biology, biodiversity, ecosystem",The biosphere refers to the sum of all ecosyst...,"biology, biodiversity, ecosystem",embedding_top3
2,3,Respiratory system,engine,"[engine, Air handling system, air circulation ...",respiration,-1,0,[],[],"[respiration, Human Body, the skeletal system]","[0.5764765716606284, 0.4765361332478554, 0.433...",name_properties,"Respiratory system. oxygen, the lungs, breathi...",The respiratory system is a critical biologica...,"oxygen, the lungs, breathing muscles",embedding_top3
3,4,Spread of Pathogens,Spread of Fire,[Spread of Fire],Crowd,3,1,[Spread of Fire],[3],"[Crowd, Disease, Spread of Fire]","[0.5004493275992501, 0.47933010292152944, 0.45...",name_properties,"Spread of Pathogens. pathogen, crowd, Preventi...","The spread of pathogens is a serious concern, ...","pathogen, crowd, Prevention and control measures",embedding_top3
4,5,Gene editing,kirigami,[kirigami],bacterial mutation,-1,0,[],[],"[bacterial mutation, Evolution, edit]","[0.37029149403542694, 0.3350655099294726, 0.32...",name_properties,"Gene editing. Gene, CRISPR-Cas9 Technology, ed...",Gene editing is a revolutionary technique in m...,"Gene, CRISPR-Cas9 Technology, edited gene",embedding_top3


## Baseline 2: Embedding Retrieval + LLM Selection (Top 10 → 3)


In [5]:
# =============================================================================
# BUILD LOOKUPS
# =============================================================================
df_scar = pd.read_csv(DATA_PATH)

# Source properties lookup
source_properties_lookup = {}
for _, row in df_scar.iterrows():
    source_name = row['system_b']
    if source_name not in source_properties_lookup:
        source_properties_lookup[source_name] = set()
    if pd.notna(row.get('mappings_parsed')):
        try:
            mappings = ast.literal_eval(row['mappings_parsed'])
            for pair in mappings:
                if len(pair) > 1:
                    source_properties_lookup[source_name].add(pair[1])
        except:
            pass
source_properties_lookup = {k: ", ".join(sorted(v)) for k, v in source_properties_lookup.items()}

# Source description lookup
source_desc_lookup = {}
for _, row in df_scar.iterrows():
    source_name = row['system_b']
    if source_name not in source_desc_lookup:
        source_desc_lookup[source_name] = row['system_b_background'] if pd.notna(row['system_b_background']) else ""

def get_source_properties(source_name):
    """Get properties for a source from lookup"""
    return source_properties_lookup.get(source_name, "")

def get_target_properties(row):
    """Extract target properties from mappings_parsed"""
    if pd.notna(row.get('mappings_parsed')):
        try:
            mappings = ast.literal_eval(row['mappings_parsed'])
            return [pair[0] for pair in mappings if len(pair) > 0]
        except:
            pass
    return []

# =============================================================================
# BASELINE CONDITIONS
# =============================================================================
class BaselineCondition:
    NAME_ONLY = "name_only"           # Just candidate name
    NAME_PROPS = "name_properties"    # Name + properties  
    NAME_DESC = "name_description"    # Name + description
    FULL = "name_props_desc"          # Name + properties + description

def format_candidate_info(name, condition):
    """Format candidate info based on baseline condition."""
    if condition == BaselineCondition.NAME_ONLY:
        return ""
    elif condition == BaselineCondition.NAME_PROPS:
        return get_source_properties(name)
    elif condition == BaselineCondition.NAME_DESC:
        desc = source_desc_lookup.get(name, "")[:200]
        return f"Description: {desc}..." if desc else ""
    elif condition == BaselineCondition.FULL:
        props = get_source_properties(name)
        desc = source_desc_lookup.get(name, "")[:200]
        return f"Properties: {props} | Description: {desc}..."
    return ""

# =============================================================================
# DSPy RERANKER (identical to rag_source_mapping_pipeline.ipynb)
# =============================================================================
RERANKER_INSTRUCTIONS = """You are selecting the best analogous source concepts for a scientific analogy.

Your task:
1. Analyze the target concept and its properties
2. Review each candidate source and its information
3. Select the 3 candidates that would make the BEST analogies for the target

Selection criteria:
- The source concept should be familiar and help explain the unfamiliar target
- Strong structural/functional correspondence between source and target
- Prefer sources that are from different domains (far-field analogies)

Return the EXACT names of your top 3 selected candidates."""


def llm_select_sources_dspy(target_name, target_info, candidates, condition, top_k=3, max_retries=3):
    """
    Unified DSPy reranker - selects by NAME with reasoning capture.
    Identical to rag_source_mapping_pipeline.ipynb.
    
    Args:
        target_name: Target concept name
        target_info: Additional context (properties string)
        candidates: List of candidate objects with .name attribute
        condition: BaselineCondition type
    
    Returns:
        tuple: (selected_names, reasoning) or (ERROR_MARKER, error_msg)
    """
    # Build candidate names and formatted text
    candidate_names = [c.name for c in candidates]
    
    candidates_text = "\n".join([
        f"- {c.name}: {format_candidate_info(c.name, condition)}"
        for c in candidates
    ])
    
    # Debug: print reranker inputs
    if DEBUG_VERBOSE:
        print(f"\n{'='*60}")
        print(f"[RERANKER INPUT]")
        print(f"  Target: {target_name}")
        print(f"  Target Info: {target_info[:100]}..." if len(target_info) > 100 else f"  Target Info: {target_info}")
        print(f"  Condition: {condition}")
        print(f"  Candidates ({len(candidates)}):")
        for c in candidates[:3]:
            print(f"    - {c.name}")
        print(f"    ...")
        print(f"{'='*60}")
    
    last_error = None
    
    for attempt in range(max_retries):
        try:
            # Create signature with instructions
            signature = dspy.Signature(
                "target_concept: str, target_info: str, candidates_info: str -> selected_sources: list[str]",
                instructions=RERANKER_INSTRUCTIONS
            )
            
            reranker = dspy.ChainOfThought(signature)
            result = reranker(
                target_concept=target_name,
                target_info=target_info,
                candidates_info=candidates_text
            )
            
            # Validate selected names exist in candidates (fuzzy match)
            selected_raw = result.selected_sources
            selected_valid = []
            
            for sel_name in selected_raw:
                # Try exact match first
                if sel_name in candidate_names:
                    selected_valid.append(sel_name)
                else:
                    # Try case-insensitive match
                    for cand_name in candidate_names:
                        if sel_name.lower() == cand_name.lower():
                            selected_valid.append(cand_name)
                            break
                        # Partial match
                        elif sel_name.lower() in cand_name.lower() or cand_name.lower() in sel_name.lower():
                            selected_valid.append(cand_name)
                            break
            
            # Debug output
            if DEBUG_VERBOSE:
                print(f"\n[RERANKER OUTPUT]")
                print(f"  Selected (raw): {selected_raw}")
                print(f"  Selected (validated): {selected_valid}")
                reasoning_preview = result.reasoning[:200] if len(result.reasoning) > 200 else result.reasoning
                print(f"  Reasoning: {reasoning_preview}...")
            
            if selected_valid:
                return (selected_valid[:top_k], result.reasoning)
            else:
                if attempt < max_retries - 1:
                    print(f"  [RERANKER] No valid selections found, retrying...")
                    time.sleep((attempt + 1) * 2)
                    
        except Exception as e:
            last_error = str(e)
            if DEBUG_VERBOSE:
                print(f"\n  [RERANKER ERROR] Attempt {attempt + 1}/{max_retries}")
                print(f"    Error: {last_error[:300]}...")
            
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
    
    return (ERROR_MARKER, last_error or "No valid selections after all retries")


# =============================================================================
# GENERIC BASELINE RUNNER
# =============================================================================
def run_baseline(df, finder, condition, baseline_name):
    """
    Generic baseline runner - eliminates repetitive loop code.
    
    Args:
        df: DataFrame with target concepts
        finder: RAGSourceFinder instance
        condition: BaselineCondition type
        baseline_name: String name for logging
    
    Returns:
        DataFrame with results
    """
    results = []
    errors = 0
    
    print(f"\n{'#'*70}")
    print(f"# Running {baseline_name} ({condition})")
    print(f"# Processing {len(df)} examples...")
    print(f"{'#'*70}\n")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=baseline_name):
        target_name = row['system_a']
        target_bg = row['system_a_background'] if pd.notna(row['system_a_background']) else ""
        target_props = get_target_properties(row)
        gold_source = row['system_b']
        
        # Get top 10 candidates from embeddings
        candidates = finder.find_source(target_name, target_bg, top_k=10)
        
        # Format target info based on condition
        if condition == BaselineCondition.NAME_ONLY:
            target_info = ""
        elif condition == BaselineCondition.NAME_PROPS:
            target_info = ", ".join(target_props)
        elif condition == BaselineCondition.NAME_DESC:
            target_info = target_bg[:300]
        elif condition == BaselineCondition.FULL:
            target_info = f"Properties: {', '.join(target_props)} | Description: {target_bg[:200]}"
        else:
            target_info = ""
        
        # Run DSPy reranker
        selected_names, reasoning = llm_select_sources_dspy(
            target_name, target_info, candidates, condition
        )
        
        # Handle errors
        if selected_names == ERROR_MARKER:
            errors += 1
            status = "error"
            gold_rank = -1
            selected_names_list = []
        else:
            status = "success"
            selected_names_list = selected_names
            
            # Find gold rank in selected
            gold_rank = -1
            for i, name in enumerate(selected_names_list):
                if gold_source.lower() in name.lower() or name.lower() in gold_source.lower():
                    gold_rank = i + 1
                    break
        
        results.append({
            'id': row['id'],
            'target': target_name,
            'gold_source': gold_source,
            'top_10_sources': [c.name for c in candidates],
            'llm_selected_sources': selected_names_list,
            'reranker_reasoning': reasoning,
            'gold_rank': gold_rank,
            'status': status,
            'condition': condition
        })
        
        # Progress update every 50
        if (idx + 1) % 50 == 0:
            found_so_far = sum(1 for r in results if r['gold_rank'] > 0)
            error_so_far = sum(1 for r in results if r['status'] == 'error')
            print(f"\nProgress: {idx+1}/{len(df)} - Hit rate: {100*found_so_far/len(results):.1f}% - Errors: {error_so_far}")
    
    print(f"\n{'='*60}")
    print(f"{baseline_name} Complete!")
    print(f"  Total: {len(results)}")
    print(f"  Errors: {errors}")
    successful = [r for r in results if r['status'] == 'success']
    if successful:
        found = sum(1 for r in successful if r['gold_rank'] > 0)
        print(f"  Hit rate (success only): {100*found/len(successful):.1f}%")
    print(f"{'='*60}\n")
    
    return pd.DataFrame(results)


print("Helper functions and DSPy reranker defined!")


Helper functions and DSPy reranker defined!


In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH)

# Run Baseline 2: LLM with Properties
baseline2_df = run_baseline(df, finder, BaselineCondition.NAME_PROPS, "Baseline 2 (Name + Properties)")
baseline2_df.to_csv('results/embedding_llm_name_subconcepts.csv', index=False)
print(f"Baseline 2 saved: {len(baseline2_df)} rows")
baseline2_df.head()



######################################################################
# Running Baseline 2 (Name + Properties) (name_properties)
# Processing 400 examples...
######################################################################



Baseline 2 (Name + Properties):   0%|          | 0/400 [00:00<?, ?it/s]


[RERANKER INPUT]
  Target: biological clock
  Target Info: changes, state, adjust
  Condition: name_properties
  Candidates (10):
    - Biological Evolution
    - clock
    - bacterial mutation
    ...


Baseline 2 (Name + Properties):   0%|          | 1/400 [00:28<3:11:27, 28.79s/it]


[RERANKER OUTPUT]
  Selected (raw): ['clock', 'day and night cycle', 'Human Body']
  Selected (validated): ['clock', 'day and night cycle', 'Human Body']
  Reasoning: To find the best analogous source concepts for the target concept "biological clock," we need to analyze the properties of the target and review each candidate source. The target concept involves chan...

[RERANKER INPUT]
  Target: Biosphere
  Target Info: biology, biodiversity, ecosystem
  Condition: name_properties
  Candidates (10):
    - Ecosystem
    - ecosystem
    - Biological Evolution
    ...


...


[RERANKER OUTPUT]
  Selected (raw): ['atom', 'electromagnetic emission system', 'Computer Processor']
  Selected (validated): ['atom', 'electromagnetic emission system', 'Computer Processor']
  Reasoning: To select the best analogous source concepts for a Proton Accelerator, we need to analyze the target concept and its properties, and then review each candidate source and its information. A Proton Acc...

[RERANKER INPUT]
  Target: Particle Accelerator
  Target Info: particle, accelerator, detector
  Condition: name_properties
  Candidates (10):
    - Reactor
    - atom
    - Shooting Through Walls
    ...


Baseline 2 (Name + Properties): 100%|██████████| 400/400 [3:21:05<00:00, 30.16s/it]


[RERANKER OUTPUT]
  Selected (raw): ['A Pinball Game', 'electromagnetic emission system', 'engine']
  Selected (validated): ['A Pinball Game', 'electromagnetic emission system', 'engine']
  Reasoning: To select the best analogous source concepts for a Particle Accelerator, we need to analyze the target concept and its properties, and then review each candidate source and its information. A Particle...

Progress: 400/400 - Hit rate: 35.8% - Errors: 0

Baseline 2 (Name + Properties) Complete!
  Total: 400
  Errors: 0
  Hit rate (success only): 35.8%



Baseline 2 saved: 400 rows


,id,target,gold_source,top_10_sources,llm_selected_sources,reranker_reasoning,gold_rank,status,condition
0,1,biological clock,clock,"[Biological Evolution, clock, bacterial mutati...","[clock, day and night cycle, Human Body]",To find the best analogous source concepts for...,1,success,name_properties
1,2,Biosphere,Library,"[Ecosystem, ecosystem, Biological Evolution, C...","[Circular Economy, Skin System, Desert]",To find the best analogous source concepts for...,-1,success,name_properties
2,3,Respiratory system,engine,"[respiration, Air handling system, Human Body,...","[Air handling system, air circulation ducts, a...",To find the best analogous source concepts for...,-1,success,name_properties
3,4,Spread of Pathogens,Spread of Fire,"[Spread of Fire, Disease, Population Movement,...","[Spread of Fire, Information Flow, Wave Propag...",To find the best analogous source concepts for...,1,success,name_properties
4,5,Gene editing,kirigami,"[bacterial mutation, edit, Evolution, program,...","[edit, Computer Code, Printing]",To select the best analogous source concepts f...,-1,success,name_properties


## Baseline 2b: LLM Selection - Target Name Only


In [ ]:
# Run Baseline 2b: LLM with Name Only
baseline2b_df = run_baseline(df, finder, BaselineCondition.NAME_ONLY, "Baseline 2b (Name Only)")
baseline2b_df.to_csv('results/embedding_llm_name_only.csv', index=False)
print(f"Baseline 2b saved: {len(baseline2b_df)} rows")



######################################################################
# Running Baseline 2b (Name Only) (name_only)
# Processing 400 examples...
######################################################################



Baseline 2b (Name Only):   0%|          | 0/400 [00:00<?, ?it/s]


[RERANKER INPUT]
  Target: biological clock
  Target Info: 
  Condition: name_only
  Candidates (10):
    - Biological Evolution
    - clock
    - bacterial mutation
    ...


Baseline 2b (Name Only):   0%|          | 1/400 [00:10<1:10:54, 10.66s/it]


[RERANKER OUTPUT]
  Selected (raw): ['clock', 'day and night cycle', 'life']
  Selected (validated): ['clock', 'day and night cycle', 'life']
  Reasoning: To select the best analogous source concepts for the target concept "biological clock," we need to analyze the properties of the biological clock and review each candidate source for strong structural...

[RERANKER INPUT]
  Target: Biosphere
  Target Info: 
  Condition: name_only
  Candidates (10):
    - Ecosystem
    - ecosystem
    - Biological Evolution
    ...


...


[RERANKER OUTPUT]
  Selected (raw): ['Sun Protection', 'Immunity', 'Greenhouse']
  Selected (validated): ['Sun Protection', 'Immunity', 'Greenhouse']
  Reasoning: To select the best analogous source concepts for Radiation Protection, we need to analyze the properties of the target concept and review each candidate source. Since Radiation Protection involves shi...

[RERANKER INPUT]
  Target: Proton Accelerator
  Target Info: 
  Condition: name_only
  Candidates (10):
    - Reactor
    - atom
    - Factory
    ...


Baseline 2b (Name Only): 100%|█████████▉| 399/400 [1:14:38<00:13, 13.49s/it]


[RERANKER OUTPUT]
  Selected (raw): ['electromagnetic emission system', 'Factory System', 'Computer Processor']
  Selected (validated): ['electromagnetic emission system', 'Factory System', 'Computer Processor']
  Reasoning: A Proton Accelerator is a complex scientific device used in physics research, particularly in high-energy physics. To find suitable analogies, we need to consider candidates that share similar structu...

[RERANKER INPUT]
  Target: Particle Accelerator
  Target Info: 
  Condition: name_only
  Candidates (10):
    - Reactor
    - atom
    - electromagnetic emission system
    ...


Baseline 2b (Name Only): 100%|██████████| 400/400 [1:14:53<00:00, 11.23s/it]


[RERANKER OUTPUT]
  Selected (raw): ['A Pinball Game', 'engine', 'electromagnetic emission system']
  Selected (validated): ['A Pinball Game', 'engine', 'electromagnetic emission system']
  Reasoning: To select the best analogous source concepts for a Particle Accelerator, we need to consider the properties and functions of a Particle Accelerator. A Particle Accelerator is a complex device that acc...

Progress: 400/400 - Hit rate: 33.2% - Errors: 1

Baseline 2b (Name Only) Complete!
  Total: 400
  Errors: 1
  Hit rate (success only): 33.3%

Baseline 2b saved: 400 rows


## Baseline 2c: LLM Selection - Target Name + Properties + Description


In [ ]:
# Run Baseline 2c: LLM with Full Info (Name + Properties + Description)
baseline2c_df = run_baseline(df, finder, BaselineCondition.FULL, "Baseline 2c (Full)")
baseline2c_df.to_csv('results/embedding_llm_full_context.csv', index=False)
print(f"Baseline 2c saved: {len(baseline2c_df)} rows")



######################################################################
# Running Baseline 2c (Full) (name_props_desc)
# Processing 400 examples...
######################################################################



Baseline 2c (Full):   0%|          | 0/400 [00:00<?, ?it/s]


[RERANKER INPUT]
  Target: biological clock
  Target Info: Properties: changes, state, adjust | Description: The biological clock is a fundamental aspect of hu...
  Condition: name_props_desc
  Candidates (10):
    - Biological Evolution
    - clock
    - bacterial mutation
    ...


Baseline 2c (Full):   0%|          | 1/400 [00:15<1:42:19, 15.39s/it]


[RERANKER OUTPUT]
  Selected (raw): ['clock', 'day and night cycle', 'Time Machine']
  Selected (validated): ['clock', 'day and night cycle', 'Time Machine']
  Reasoning: To find the best analogous source concepts for the target concept "biological clock," we need to analyze the properties and description of the target and compare them with the candidates. The biologic...

[RERANKER INPUT]
  Target: Biosphere
  Target Info: Properties: biology, biodiversity, ecosystem | Description: The biosphere refers to the sum of all e...
  Condition: name_props_desc
  Candidates (10):
    - Ecosystem
    - ecosystem
    - Biological Evolution
    ...


...


[RERANKER OUTPUT]
  Selected (raw): ['Sun Protection', 'Greenhouse', 'Immunity']
  Selected (validated): ['Sun Protection', 'Greenhouse', 'Immunity']
  Reasoning: To select the best analogous source concepts for Radiation Protection, we need to analyze the properties and descriptions of each candidate source. The target concept, Radiation Protection, involves s...

[RERANKER INPUT]
  Target: Proton Accelerator
  Target Info: Properties: proton, accelerator, target | Description: A proton accelerator is a type of particle ac...
  Condition: name_props_desc
  Candidates (10):
    - Reactor
    - atom
    - Factory
    ...


Baseline 2c (Full): 100%|█████████▉| 399/400 [1:28:07<00:11, 11.03s/it]


[RERANKER OUTPUT]
  Selected (raw): ['Reactor', 'electromagnetic emission system', 'Computer Processor']
  Selected (validated): ['Reactor', 'electromagnetic emission system', 'Computer Processor']
  Reasoning: To select the best analogous source concepts for the Proton Accelerator, we need to analyze its properties and description. A proton accelerator is a device that uses electromagnetic fields to acceler...

[RERANKER INPUT]
  Target: Particle Accelerator
  Target Info: Properties: particle, accelerator, detector | Description: A particle accelerator is a powerful mach...
  Condition: name_props_desc
  Candidates (10):
    - Reactor
    - atom
    - Shooting Through Walls
    ...


Baseline 2c (Full): 100%|██████████| 400/400 [1:28:21<00:00, 13.25s/it]


[RERANKER OUTPUT]
  Selected (raw): ['A Pinball Game', 'electromagnetic emission system', 'engine']
  Selected (validated): ['A Pinball Game', 'electromagnetic emission system', 'engine']
  Reasoning: To select the best analogous source concepts for the target concept "Particle Accelerator", we need to analyze the properties and description of the target concept and review each candidate source. Th...

Progress: 400/400 - Hit rate: 33.8% - Errors: 0

Baseline 2c (Full) Complete!
  Total: 400
  Errors: 0
  Hit rate (success only): 33.8%

Baseline 2c saved: 400 rows


## Baseline 2d: LLM Selection - Target Name + Description


In [ ]:
# Run Baseline 2d: LLM with Name + Description
baseline2d_df = run_baseline(df, finder, BaselineCondition.NAME_DESC, "Baseline 2d (Name + Description)")
baseline2d_df.to_csv('results/embedding_llm_name_desc.csv', index=False)
print(f"Baseline 2d saved: {len(baseline2d_df)} rows")



######################################################################
# Running Baseline 2d (Name + Description) (name_description)
# Processing 400 examples...
######################################################################



Baseline 2d (Name + Description):   0%|          | 0/400 [00:00<?, ?it/s]


[RERANKER INPUT]
  Target: biological clock
  Target Info: The biological clock is a fundamental aspect of human physiology that governs the body's internal rh...
  Condition: name_description
  Candidates (10):
    - Biological Evolution
    - clock
    - bacterial mutation
    ...


Baseline 2d (Name + Description):   0%|          | 1/400 [00:14<1:33:52, 14.12s/it]


[RERANKER OUTPUT]
  Selected (raw): ['clock', 'day and night cycle', 'wrinkles']
  Selected (validated): ['clock', 'day and night cycle', 'wrinkles']
  Reasoning: To select the best analogous source concepts for the biological clock, we need to analyze the target concept and its properties, and then review each candidate source and its information. The biologic...

[RERANKER INPUT]
  Target: Biosphere
  Target Info: The biosphere refers to the sum of all ecosystems on Earth, including the interactions between livin...
  Condition: name_description
  Candidates (10):
    - Ecosystem
    - ecosystem
    - Biological Evolution
    ...


...


[RERANKER OUTPUT]
  Selected (raw): ['Sun Protection', 'Immunity', 'Greenhouse']
  Selected (validated): ['Sun Protection', 'Immunity', 'Greenhouse']
  Reasoning: To select the best analogous source concepts for Radiation Protection, we need to analyze the target concept and its properties, and then review each candidate source and its information. Radiation pr...

[RERANKER INPUT]
  Target: Proton Accelerator
  Target Info: A proton accelerator is a type of particle accelerator that uses electromagnetic fields to accelerat...
  Condition: name_description
  Candidates (10):
    - Reactor
    - atom
    - Factory
    ...


Baseline 2d (Name + Description): 100%|█████████▉| 399/400 [1:15:48<00:16, 16.15s/it]


[RERANKER OUTPUT]
  Selected (raw): ['Reactor', 'Factory', 'Computer Processor']
  Selected (validated): ['Reactor', 'Factory', 'Computer Processor']
  Reasoning: To select the best analogous source concepts for a Proton Accelerator, we need to analyze the target concept and its properties, and then review each candidate source and its information. A Proton Acc...

[RERANKER INPUT]
  Target: Particle Accelerator
  Target Info: A particle accelerator is a powerful machine that uses electromagnetic fields to propel charged part...
  Condition: name_description
  Candidates (10):
    - Reactor
    - atom
    - Shooting Through Walls
    ...


Baseline 2d (Name + Description): 100%|██████████| 400/400 [1:15:55<00:00, 11.39s/it]


[RERANKER OUTPUT]
  Selected (raw): ['A Pinball Game', 'engine', 'electromagnetic emission system']
  Selected (validated): ['A Pinball Game', 'engine', 'electromagnetic emission system']
  Reasoning: To select the best analogous source concepts for the target concept "Particle Accelerator", we need to analyze the properties of a particle accelerator and find sources that have similar characteristi...

Progress: 400/400 - Hit rate: 35.0% - Errors: 0

Baseline 2d (Name + Description) Complete!
  Total: 400
  Errors: 0
  Hit rate (success only): 35.0%

Baseline 2d saved: 400 rows


## Summary Statistics


In [ ]:
# =============================================================================
# BASELINE COMPARISON (with error tracking)
# =============================================================================

def compute_baseline_stats(df, name, has_status=True):
    """Compute stats for a baseline, handling error cases."""
    total = len(df)
    
    if has_status and 'status' in df.columns:
        successful = df[df['status'] == 'success']
        errors = len(df[df['status'] == 'error'])
    else:
        successful = df
        errors = 0
    
    successful_count = len(successful)
    found = (successful['gold_rank'] > 0).sum()
    hit_rate = 100 * found / successful_count if successful_count > 0 else 0
    mean_rank = successful[successful['gold_rank'] > 0]['gold_rank'].mean()
    
    return {
        'baseline': name,
        'total': total,
        'successful': successful_count,
        'errors': errors,
        'found': found,
        'hit_rate': hit_rate,
        'mean_rank': mean_rank
    }

print("=" * 70)
print("BASELINE COMPARISON")
print("=" * 70)

results_summary = []

# Baseline 1: Embedding (no status column)
stats = compute_baseline_stats(baseline1_results, 'Embedding Top 3', has_status=False)
results_summary.append(stats)
print(f"\n1. {stats['baseline']}:")
print(f"   Gold found: {stats['found']}/{stats['total']} ({stats['hit_rate']:.1f}%)")
print(f"   Mean gold rank (when found): {stats['mean_rank']:.2f}")

# Baseline 2: LLM + Properties
stats = compute_baseline_stats(baseline2_df, 'LLM + Properties')
results_summary.append(stats)
print(f"\n2. {stats['baseline']}:")
print(f"   Gold found: {stats['found']}/{stats['successful']} ({stats['hit_rate']:.1f}%)")
if stats['errors'] > 0:
    print(f"   Errors: {stats['errors']}")
print(f"   Mean gold rank (when found): {stats['mean_rank']:.2f}")

# Baseline 2b: LLM + Name Only
stats = compute_baseline_stats(baseline2b_df, 'LLM + Name Only')
results_summary.append(stats)
print(f"\n2b. {stats['baseline']}:")
print(f"   Gold found: {stats['found']}/{stats['successful']} ({stats['hit_rate']:.1f}%)")
if stats['errors'] > 0:
    print(f"   Errors: {stats['errors']}")
print(f"   Mean gold rank (when found): {stats['mean_rank']:.2f}")

# Baseline 2c: LLM + Full
stats = compute_baseline_stats(baseline2c_df, 'LLM: Name + Props + Desc')
results_summary.append(stats)
print(f"\n2c. {stats['baseline']}:")
print(f"   Gold found: {stats['found']}/{stats['successful']} ({stats['hit_rate']:.1f}%)")
if stats['errors'] > 0:
    print(f"   Errors: {stats['errors']}")
print(f"   Mean gold rank (when found): {stats['mean_rank']:.2f}")

# Baseline 2d: LLM + Name + Desc
stats = compute_baseline_stats(baseline2d_df, 'LLM: Name + Desc')
results_summary.append(stats)
print(f"\n2d. {stats['baseline']}:")
print(f"   Gold found: {stats['found']}/{stats['successful']} ({stats['hit_rate']:.1f}%)")
if stats['errors'] > 0:
    print(f"   Errors: {stats['errors']}")
print(f"   Mean gold rank (when found): {stats['mean_rank']:.2f}")

print("\n" + "=" * 70)

# Save summary
summary_df = pd.DataFrame(results_summary)
summary_df.to_csv('results/baselines_comparison_summary.csv', index=False)
print("\nSummary saved to results/baselines_comparison_summary.csv")
summary_df


BASELINE COMPARISON

1. Embedding Top 3:
   Gold found: 160/400 (40.0%)
   Mean gold rank (when found): 1.75

2. LLM + Properties:
   Gold found: 143/400 (35.8%)
   Mean gold rank (when found): 1.43

2b. LLM + Name Only:
   Gold found: 133/399 (33.3%)
   Errors: 1
   Mean gold rank (when found): 1.59

2c. LLM: Name + Props + Desc:
   Gold found: 135/400 (33.8%)
   Mean gold rank (when found): 1.50

2d. LLM: Name + Desc:
   Gold found: 140/400 (35.0%)
   Mean gold rank (when found): 1.51


Summary saved to baselines_summary.csv


,baseline,total,successful,errors,found,hit_rate,mean_rank
0,Embedding Top 3,400,400,0,160,40.000000,1.750000
1,LLM + Properties,400,400,0,143,35.750000,1.426573
2,LLM + Name Only,400,399,1,133,33.333333,1.586466
3,LLM: Name + Props + Desc,400,400,0,135,33.750000,1.503704
4,LLM: Name + Desc,400,400,0,140,35.000000,1.514286
